# Testing validation codes

In [1]:
import sys
import os

# Go from notebooks/test_stuff → meta_analysis_agents
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

from src.experimentutils.eval_utils import (
    evaluate_record_pair_with_units,
    field_similarity_score,
)
from src.experimentutils.units import unit, Unit, Quantity, UnitStorage

In [2]:
# field_similarity_score("2", "2")
# field_similarity_score("1987-88", "1987")
field_similarity_score("35514", "1997-03-25", field_name="Harvest date 1")  # -> 1.0 (if serial maps to same date)


1.0

In [3]:
field_similarity_score("35514", "25 March", field_name="Sowing date 1")


1.0

In [4]:
# unit tests
q1 = 50 * unit("kg ha-1")
q2 = 50000 * unit("g ha-1")
assert q1 == q2

assert 1 * unit("t/ha") == 1 * unit("Mg/ha")
assert 10 * unit("kg/ha") == 1 * unit("g/m2")
assert 100 * unit("kg N ha-1") == 10 * unit("g N m-2")
print("unit arithmetic: ok")

unit arithmetic: ok


## Two-record evaluation (`evaluate_record_pair_with_units`)

Compare one **prediction** row vs one **ground-truth** row (dicts). Steps: per-field scores → value+unit where applicable → try crop 1/2 swap on prediction and keep the better mean score.

In [5]:
# Shared fields for a minimal wopke-style record
FIELDS = [
    "Crop species 1",
    "Crop species 2",
    "N input SC1",
    "N Unit",
    "unified yield sc 1",
    "Yield unit",
    "Year of data",
]

# --- Test 1: value+unit match (50 kg ha-1 == 50000 g ha-1), crops aligned ---
pred_1 = {
    "Crop species 1": "wheat",
    "Crop species 2": "bean",
    "N input SC1": "50",
    "N Unit": "kg ha-1",
    "unified yield sc 1": "8.5",
    "Yield unit": "t/ha",
    "Year of data": "1987",
}

gt_1 = {
    "Crop species 1": "wheat",
    "Crop species 2": "bean",
    "N input SC1": "50000",
    "N Unit": "g ha-1",
    "unified yield sc 1": "8.5",
    "Yield unit": "t/ha",
    "Year of data": "1987-88",
}

result_1 = evaluate_record_pair_with_units(pred_1, gt_1, FIELDS)

print("Test 1 — value+unit + year range")
print(f"  swapped: {result_1.swapped}")
print(f"  mean (original): {result_1.mean_score_original:.3f}")
print(f"  mean (swapped):  {result_1.mean_score_swapped:.3f}")
print(f"  mean (final):    {result_1.mean_score:.3f}")
print("  per-field (final):")
for f, s in result_1.field_scores.items():
    print(f"    {f:22s} {s:.3f}")

assert result_1.field_scores["N input SC1"] == 1.0
assert result_1.field_scores["N Unit"] == 1.0  # same as value when Quantity matches
assert result_1.field_scores["Yield unit"] == 1.0
assert result_1.field_scores["Year of data"] == 1.0
assert result_1.swapped is False
print("\nTest 1 passed.")

Test 1 — value+unit + year range
  swapped: False
  mean (original): 1.000
  mean (swapped):  0.841
  mean (final):    1.000
  per-field (final):
    Crop species 1         1.000
    Crop species 2         1.000
    N input SC1            1.000
    unified yield sc 1     1.000
    Year of data           1.000
    N Unit                 1.000
    Yield unit             1.000

Test 1 passed.


In [6]:
# --- Test 2: crops reversed on prediction → swap should win ---
pred_2 = {
    "Crop species 1": "bean",   # reversed vs GT
    "Crop species 2": "wheat",
    "N input SC1": "50",
    "N Unit": "kg ha-1",
    "unified yield sc 1": "8.5",
    "Yield unit": "t/ha",
    "Year of data": "1987",
}

gt_2 = dict(gt_1)  # same GT as test 1

result_2 = evaluate_record_pair_with_units(pred_2, gt_2, FIELDS)

print("Test 2 — crop label swap")
print(f"  swapped: {result_2.swapped}")
print(f"  mean (original): {result_2.mean_score_original:.3f}")
print(f"  mean (swapped):  {result_2.mean_score_swapped:.3f}")
print(f"  mean (final):    {result_2.mean_score:.3f}")
print("  species scores (original vs swapped vs final):")
for label, scores in [
    ("original", result_2.field_scores_original),
    ("swapped", result_2.field_scores_swapped),
    ("final", result_2.field_scores),
]:
    print(
        f"    [{label}] species1={scores['Crop species 1']:.3f} "
        f"species2={scores['Crop species 2']:.3f}"
    )

assert result_2.swapped is True
assert result_2.mean_score_swapped > result_2.mean_score_original
assert result_2.field_scores["Crop species 1"] == 1.0
assert result_2.field_scores["Crop species 2"] == 1.0
assert result_2.field_scores["N input SC1"] == 1.0
print("\nTest 2 passed.")

Test 2 — crop label swap
  swapped: True
  mean (original): 0.841
  mean (swapped):  1.000
  mean (final):    1.000
  species scores (original vs swapped vs final):
    [original] species1=0.444 species2=0.444
    [swapped] species1=1.000 species2=1.000
    [final] species1=1.000 species2=1.000

Test 2 passed.
